## WWF Priority Place — 10-Class Classifier

### 1 · Imports

In [1]:
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
WWF_GREEN  = '#006B3C'
WWF_GOLD   = '#F5A623'
AMAZON_CLR = '#E8742A'
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9'})

### 2 · Keyword Config

In [2]:
# Segment review order per Training_Rules.pdf
SEGMENT_ORDER = [
    'Cost Center',   # Level 1
    'Program Code',  # Level 2
    'Grant',         # Level 3
]

# Used only at inference time — Country is too ambiguous for training data labeling
# (e.g. Colombia appears in both Amazon and Eastern Pacific)
INFERENCE_SEGMENT_ORDER = [
    'Cost Center',
    'Program Code',
    'Grant',
    'Country',
]

PRIORITY_PLACE_KEYWORDS = {
    'Amazon': {
        'countries': [
            'brazil', 'brasil', 'bolivia', 'bolivie', 'peru', 'perú', 'ecuador',
            'équateur', 'colombia', 'colombie', 'guyana', 'suriname', 'surinam',
            'french guiana', 'guyane',
        ],
        'geo_features': [
            'amazon', 'amazonia', 'amazônia', 'amazonica', 'amazónica',
            'tapajos', 'tapajós', 'madeira', 'xingu', 'rio negro', 'solimões',
            'solimoes', 'marañón', 'maranon', 'ucayali', 'napo river', 'putumayo',
            'jurua', 'juruá', 'purus', 'purús', 'javari', 'japura', 'japurá',
            'madre de dios', 'río beni', 'guapore', 'guaporé', 'vaupés', 'vaupes',
            'içá', 'ica river', 'southwest amazon', 'south amazon', 'western amazon',
            'amazon fire', 'amazon fires', 'heco', 'leticia', 'jaguar corridor',
        ],
        'protected_areas': [
            'manu national park', 'yasuni', 'yasuní', 'madidi', 'tambopata', 'pacaya-samiria',
            'pacaya samiria', 'serra do divisor', 'sierra del divisor',
            'jau national park', 'jaú', 'tumucumaque', 'alto purus', 'alto purús',
            'chiribiquete', 'cordillera azul', 'amboro', 'amboró',
            'beni biosphere', 'pantanal matogrossense',
        ],
        'species': [
            'jaguar', 'onca', 'onça', 'giant otter', 'giant river otter',
            'ariranha', 'tapir', 'pink river dolphin', 'boto', 'bufeo',
            'arapaima', 'paiche', 'pirarucu', 'macaw', 'arara', 'harpy eagle',
            'gavião-real', 'giant anteater', 'tamandua', 'giant armadillo',
            'tatu-canastra', 'anaconda', 'sucuri', 'black caiman', 'jacare',
            'jacaré', 'arrau turtle', 'capybara', 'capivara', 'manatee',
            'peixe-boi', 'howler monkey', 'spider monkey', 'woolly monkey',
            'uakari', 'squirrel monkey', 'poison dart frog', 'piranha',
        ],
    },

    'Congo Basin': {
        'countries': [
            'drc', 'democratic republic of congo', 'democratic republic of the congo',
            'dr congo', 'congo', 'kinshasa', 'republic of congo', 'republic of the congo',
            'brazzaville', 'central african republic', 'gabon', 'cameroon',
            'cameroun', 'equatorial guinea', 'guinée équatoriale',
        ],
        'geo_features': [
            'congo river', 'congo basin', 'congo rainforest', 'ubangi', 'sangha',
            'kasai', 'lualaba', 'uele', 'aruwimi', 'lake tumba', 'lake mai-ndombe',
            'cuvette centrale', 'cuvette', 'basin du congo', 'bassin du congo',
            'lobaye', 'ituri', 'kwilu', 'kwango',
        ],
        'protected_areas': [
            'salonga', 'virunga', 'dja', 'odzala', 'nouabale-ndoki', 'nouabalé-ndoki',
            'dzanga-sangha', 'dzanga sangha', 'lope', 'lopé', 'minkebe',
            'tridom', 'ntokou-pikounda', 'ntokou pikounda', 'conkouati-douli',
            'mbam et djerem', 'boumba-bek', 'boumba bek', 'campo maan',
            'lobeke', 'lobéké', 'odzala-kokoua', 'cbfp', 'congo basin forest partnership',
        ],
        'species': [
            'gorilla', 'gorille', 'chimpanzee', 'chimpanze', 'bonobo',
            'forest elephant', 'éléphant de forêt', 'okapi', 'bongo',
            'congo peacock', 'paon du congo', 'drill', 'mandrill',
            'red colobus', 'colobus', 'golden potto', 'african forest buffalo',
            'sitatunga', 'bongo antelope', 'pangolin',
        ],
    },

    'Eastern Himalayas': {
        'countries': [
            'bhutan', 'bhoutan', 'druk yul', 'india', 'bharat', 'hindustan',
            'nepal', 'népal',
        ],
        'geo_features': [
            'himalayas', 'himalaya', 'brahmaputra', 'yarlung tsangpo', 'tsangpo',
            'ganges', 'ganga', 'sundarbans', 'terai', 'eastern himalayas',
            'hindu kush', 'arunachal', 'sikkim', 'assam', 'meghalaya',
            'darjeeling', 'bhutan highlands',
        ],
        'protected_areas': [
            'royal manas', 'jigme dorji', 'chitwan', 'sagarmatha', 'kangchenjunga',
            'kanchenjunga', 'namdapha', 'kaziranga', 'manas', 'pakke',
            'bumdeling', 'thrumshingla', 'wangchuck centennial', 'bardia',
            'makalu-barun', 'makalu barun', 'langtang', 'annapurna',
            'phrumsengla', 'sakteng', 'bhutan for life', 'bfl',
        ],
        'species': [
            'snow leopard', 'irbis', 'red panda', 'firefox', 'ailurus',
            'bengal tiger', 'eastern himalayan tiger', 'tigers eastern himalayas', 'one-horned rhino', 'indian rhinoceros', 'greater one-horned',
            'clouded leopard', 'golden langur', 'ganges river dolphin',
            'gangetic dolphin', 'platanista', 'black-necked crane',
            'bengal florican', 'gharial', 'king cobra', 'himalayan brown bear',
            'asiatic black bear', 'takin', 'gaur', 'wild water buffalo',
        ],
    },

    'Eastern Pacific Seascape': {
        'countries': [
            'colombia', 'ecuador', 'panama', 'panamá', 'costa rica', 'chile',
        ],
        'geo_features': [
            'eastern tropical pacific', 'eastern pacific', 'cmar',
            'humboldt current', 'galápagos', 'galapagos', 'cocos island',
            'cocos', 'coiba', 'malpelo', 'pacific coast', 'pacific ocean',
            'exclusive economic zone', 'eez', 'pacific seascape',
            'chilean seascapes', 'valdivian',
        ],
        'protected_areas': [
            'galápagos national park', 'galapagos national park', 'galapagos marine reserve',
            'cocos island national park', 'coiba national park',
            'malpelo fauna and flora sanctuary', 'machalilla', 'gorgona',
            'las gemelas', 'darien', 'darién', 'uramba bahia malaga',
        ],
        'species': [
            'marine iguana', 'galápagos tortoise', 'galapagos tortoise',
            'blue-footed booby', 'galápagos penguin', 'whale shark',
            'manta ray', 'hammerhead shark', 'leatherback turtle',
            'hawksbill turtle', 'humpback whale', 'sperm whale',
            'bottlenose dolphin', 'sea turtle', 'mobula', 'sailfish',
        ],
    },

    'Great Plains': {
        'countries': [
            'united states', 'usa', 'u.s.a', 'u.s.', 'america',
            'canada', 'canadá',
        ],
        'geo_features': [
            'great plains', 'northern great plains', 'ngp', 'high plains', 'plowprint',
            'rio grande', 'rio bravo', 'usbr', 'rgrb',
            'llano estacado', 'chihuahuan desert', 'missouri river',
            'platte river', 'yellowstone river', 'prairie', 'grassland',
            'montana', 'wyoming', 'north dakota', 'south dakota',
            'nebraska', 'kansas', 'oklahoma', 'saskatchewan',
            'alberta', 'manitoba', 'sonora', 'chihuahua', 'coahuila',
        ],
        'protected_areas': [
            'yellowstone', 'badlands', 'theodore roosevelt', 'wind cave',
            'grasslands national park', 'tallgrass prairie', 'little missouri',
            'american prairie', 'charles m. russell', 'fort peck',
            'rocky mountain arsenal', 'northern great plains program',
            'wolakota', 'tribal buffalo', 'rsvp', 'pine ridge', 'fort belknap',
        ],
        'species': [
            'bison', 'american bison', 'buffalo', 'black-footed ferret',
            'prairie dog', 'pronghorn', 'swift fox', 'sage grouse',
            'burrowing owl', 'mountain plover', 'ferruginous hawk',
            'long-billed curlew', 'elk', 'wapiti', 'mule deer',
            'grizzly bear', 'gray wolf', 'wolverine', 'swift fox',
        ],
    },

    'Arctic': {
        'countries': [
            'united states', 'usa', 'alaska', 'canada', 'norway', 'norge',
            'denmark', 'danmark', 'greenland', 'grønland', 'kalaallit nunaat',
            'iceland', 'ísland', 'finland', 'suomi', 'sweden', 'sverige',
            'russia', 'rossiya',
        ],
        'geo_features': [
            'arctic', 'arctic ocean', 'beaufort sea', 'chukchi sea',
            'bering sea', 'barents sea', 'greenland sea', 'norwegian sea',
            'arctic circle', 'tundra', 'permafrost', 'svalbard', 'spitsbergen',
            'baffin island', 'banks island', 'devon island', 'ellesmere island',
            'northwest territories', 'nunavut', 'yukon', 'lapland',
            'bering', 'bering strait', 'arctic shipping',
        ],
        'protected_areas': [
            'arctic national wildlife refuge', 'anwr', 'svalbard global seed vault',
            'wrangel island', 'northeast greenland national park',
            'thelon wildlife sanctuary',
        ],
        'species': [
            'polar bear', 'ursus maritimus', 'walrus', 'odobenus rosmarus',
            'narwhal', 'monodon', 'beluga', 'beluga whale', 'delphinapterus',
            'arctic fox', 'alopex lagopus', 'caribou', 'reindeer', 'rangifer',
            'bowhead whale', 'balaena mysticetus', 'ringed seal', 'pusa hispida',
            'bearded seal', 'musk ox', 'ovibos moschatus', 'snowy owl',
            'arctic tern', 'ivory gull',
        ],
    },

    'Greater Mekong': {
        'countries': [
            'cambodia', 'cambodge', 'kampuchea', 'laos', 'lao pdr', 'lao',
            'myanmar', 'burma', 'birmanie', 'thailand', 'thaïlande', 'thai',
            'vietnam', 'viet nam', 'việt nam', 'indochina', 'indochinese',
        ],
        'geo_features': [
            'mekong', 'mekong river', 'mekong delta', 'lancang', 'irrawaddy',
            'ayeyarwady', 'salween', 'thanlwin', 'chao phraya', 'tonle sap',
            'red river', 'song hong', 'mekong basin', 'greater annamites',
            'dawna tenasserim', 'dawna-tenasserim', 'cardamom mountains',
        ],
        'protected_areas': [
            'cardamom', 'greater annamites', 'dawna tenasserim', 'cat tien',
            'phong nha', 'ke bang', 'phong nha-ke bang', 'pu mat', 'nakai nam theun',
            'xe pian', 'mondulkiri', 'virachey', 'hkakabo razi', 'hukaung valley',
            'inlay lake', 'alaungdaw kathapa', 'indawgyi lake',
        ],
        'species': [
            'mekong giant catfish', 'pangasianodon', 'irrawaddy dolphin',
            'orcaella', 'asian elephant', 'indochinese tiger', 'mekong tiger', 'tigers greater mekong', 'indochinese leopard',
            'saola', 'pseudoryx', 'green peafowl', 'banteng', 'gaur',
            'siamese crocodile', 'giant ibis', 'white-shouldered ibis',
            'fishing cat', 'clouded leopard', 'red-shanked douc',
            'sunda pangolin', 'malayan tapir',
        ],
    },

    'Southern Africa': {
        'countries': [
            'angola', 'namibia', 'zambia', 'zimbabwe', 'botswana', 'south africa',
            'suid-afrika', 'afrique du sud',
        ],
        'geo_features': [
            'zambezi', 'zambezi river', 'okavango', 'kavango', 'limpopo',
            'chobe river', 'victoria falls', 'mosi-oa-tunya', 'kalahari',
            'caprivi strip', 'cuando cubango', 'kafue', 'kavango zambezi', 'luangwa', 'kariba',
        ],
        'protected_areas': [
            'kavango zambezi transfrontier', 'okavango delta',
            'hwange', 'chobe', 'kruger', 'etosha', 'kafue', 'lower zambezi',
            'gonarezhou', 'mana pools', 'bwabwata', 'mudumu', 'sioma ngwezi',
            'luengue-luiana', 'bangweulu', 'south luangwa', 'north luangwa',
            'moremi', 'central kalahari',
        ],
        'species': [
            'african elephant', 'loxodonta', 'african lion', 'panthera leo', 'leopard',
            'cheetah', 'acinonyx', 'white rhino', 'black rhino', 'rhinoceros',
            'african wild dog', 'lycaon pictus', 'painted dog', 'hippo',
            'hippopotamus', 'giraffe', 'buffalo', 'african buffalo', 'hyena',
            'spotted hyena', 'pangolin', 'sable antelope', 'roan antelope',
            'lechwe', 'sitatunga', 'wild dog',
        ],
    },

    'Southwest Indian Ocean': {
        'countries': [
            'madagascar', 'malagasy', 'malgache', 'mozambique', 'moçambique',
            'tanzania', 'tanzanie', 'timor leste', 'timor-leste', 'east timor',
            'mauritius', 'île maurice', 'comoros', 'comores', 'seychelles',
            'reunion', 'réunion', 'la réunion',
        ],
        'geo_features': [
            'mozambique channel', 'indian ocean', 'southwest indian ocean', 'swio',
            'malagasy', 'tsiribihina', 'manambolo', 'betsiboka', 'rufiji',
            'ruvuma', 'rovuma', 'northern mozambique', 'comoro islands',
            'mascarene islands',
        ],
        'protected_areas': [
            'quirimbas', 'bazaruto', 'tsingy de bemaraha', 'tsingy',
            'ranomafana', 'isalo', 'andasibe', 'masoala', 'belo sur tsiribihina',
            'pemba', 'mafia island', 'mnazi bay', 'selous', 'niassa',
            'gorongosa', 'zinave', 'chimanimani',
        ],
        'species': [
            'lemur', 'ring-tailed lemur', 'aye-aye', 'daubentonia', 'indri lemur',
            'sifaka', 'propithecus', 'fossa', 'cryptoprocta', 'tenrec',
            'coelacanth', 'latimeria', 'dugong', 'dugong dugon', 'hawksbill',
            'green turtle', 'humpback whale', 'whale shark', 'manta ray',
            'comet grouper', 'napoleon wrasse',
        ],
    },

    'SW Pacific + Indonesia': {
        'countries': [
            'indonesia', 'indonésie', 'indonesian', 'philippines', 'pilipinas',
            'filipino', 'philippine', 'papua new guinea', 'png', 'papua',
            'fiji', 'micronesia', 'melanesia',
        ],
        'geo_features': [
            'coral triangle', 'borneo', 'sumatra', 'sumatera', 'java island', 'jawa',
            'sulawesi', 'celebes', 'papua', 'kalimantan', 'irian jaya',
            'banda sea', 'bismarck sea', 'solomon sea', 'sulu sea',
            'celebes sea', 'new guinea', 'melanesia', 'wallacea',
            'heart of borneo', 'raja ampat', 'bird\'s head seascape',
        ],
        'protected_areas': [
            'raja ampat', 'komodo', 'lorentz', 'tubbataha', 'apo reef',
            'heart of borneo', 'gunung leuser', 'leuser', 'tanjung puting',
            'thirty hills', 'kei islands',
            'kutai', 'betung kerihun', 'kayan mentarang', 'crocker range',
            'danum valley', 'kinabalu', 'coral triangle initiative',
        ],
        'species': [
            'orangutan', 'pongo', 'sumatran tiger', 'sumatran rhino',
            'dicerorhinus', 'pygmy elephant', 'bornean pygmy elephant',
            'komodo dragon', 'varanus komodoensis', 'bird of paradise',
            'paradisaea', 'cassowary', 'proboscis monkey', 'nasalis',
            'bornean orangutan', 'sumatran orangutan', 'sun bear', 'helarctos',
            'clouded leopard', 'babirusa', 'anoa', 'maleo', 'irrawaddy dolphin',
        ],
    },
}

print(f"Loaded lookup tables for {len(PRIORITY_PLACE_KEYWORDS)} priority places:")
for pp in PRIORITY_PLACE_KEYWORDS:
    total = sum(len(v) for v in PRIORITY_PLACE_KEYWORDS[pp].values())
    print(f"  {pp}: {total} keywords")

# ── Canonical place names & label mapping ─────────────────────────────────────
CANONICAL_PLACES = [
    'Amazon',
    'Arctic',
    'Congo Basin',
    'Eastern Himalayas',
    'Eastern Pacific Seascape',
    'Great Plains',
    'Greater Mekong',
    'Southern Africa',
    'Southwest Indian Ocean',
    'SW Pacific + Indonesia',
]

PLACE_TO_LABEL = {place: i for i, place in enumerate(CANONICAL_PLACES)}
LABEL_TO_PLACE = {i: place for i, place in enumerate(CANONICAL_PLACES)}

MULTI_LABEL_MARKER = '__multi__'

# Maps every raw 'Priority Place' value (after .strip()) to a canonical name.
# Multi-label rows are marked for exclusion; any future unknown value returns NaN.
PLACE_MAPPING = {
    # Clean single-label values
    'Amazon':                                    'Amazon',
    'Great Plains':                              'Great Plains',
    'Southern Africa':                           'Southern Africa',
    'Arctic':                                    'Arctic',
    'Greater Mekong':                            'Greater Mekong',
    # Typo / abbreviation fixes
    'Eastern Himalaya':                          'Eastern Himalayas',
    'Congo':                                     'Congo Basin',
    'SWIO':                                      'Southwest Indian Ocean',
    'Eastern Pacific':                           'Eastern Pacific Seascape',
    'Indonesia and Southern Pacific Islands':    'SW Pacific + Indonesia',
    # Multi-label rows — excluded from training, tracked
    'Amazon, SWIO':                                              MULTI_LABEL_MARKER,
    'SWIO, Eastern Pacific':                                     MULTI_LABEL_MARKER,
    'Eastern Pacific,SW Pacific + Indonesia, SWIO':              MULTI_LABEL_MARKER,
    'Indonesia and Southern Pacific Islands, Greater Mekong':    MULTI_LABEL_MARKER,
}

_unmapped = [v for v in PLACE_MAPPING.values() if v != MULTI_LABEL_MARKER and v not in PLACE_TO_LABEL]
assert not _unmapped, f"PLACE_MAPPING contains unknown canonical names: {_unmapped}"

print(f"Loaded {len(CANONICAL_PLACES)} canonical priority places.")
print(f"PLACE_MAPPING covers {len(PLACE_MAPPING)} raw values ({sum(v == MULTI_LABEL_MARKER for v in PLACE_MAPPING.values())} multi-label).")


Loaded lookup tables for 10 priority places:
  Amazon: 113 keywords
  Congo Basin: 78 keywords
  Eastern Himalayas: 72 keywords
  Eastern Pacific Seascape: 51 keywords
  Great Plains: 72 keywords
  Arctic: 73 keywords
  Greater Mekong: 73 keywords
  Southern Africa: 68 keywords
  Southwest Indian Ocean: 70 keywords
  SW Pacific + Indonesia: 76 keywords
Loaded 10 canonical priority places.
PLACE_MAPPING covers 14 raw values (4 multi-label).


### 3 · Pipeline 1 — Rule-Based Keyword Scan

In [3]:
def rule_scan_row(row, place_keywords=PRIORITY_PLACE_KEYWORDS, segment_order=SEGMENT_ORDER):
    for segment in segment_order:
        text = str(row.get(segment) or '').lower()
        for place, categories in place_keywords.items():
            all_keywords = [kw for kws in categories.values() for kw in kws]
            if any(kw in text for kw in all_keywords):
                return place
    return None

def apply_rules(df):
    return df.apply(rule_scan_row, axis=1)

### 4 · Load Data & Build Training Set

In [4]:
raw = pd.read_csv('Model Training Data.csv', encoding='latin1')
raw.columns = raw.columns.str.strip()
raw.drop(columns=['Unnamed: 7', 'Unnamed: 8'], inplace=True, errors='ignore')

pp_clean = raw['Priority Place'].str.strip().map(PLACE_MAPPING)

multi_label_mask = pp_clean == MULTI_LABEL_MARKER
raw['multi_label'] = multi_label_mask
print(f"Multi-label rows excluded: {multi_label_mask.sum()}")
print(raw.loc[multi_label_mask, 'Priority Place'].value_counts().to_string())
print()

raw['label'] = pp_clean.map(
    lambda p: PLACE_TO_LABEL[p] if isinstance(p, str) and p in PLACE_TO_LABEL else -1
)
train_df = raw[raw['label'] != -1].copy()

print("Pre-tagged label distribution:")
for lbl, place in sorted(LABEL_TO_PLACE.items()):
    n = int((train_df['label'] == lbl).sum())
    if n: print(f"  {place}: {n}")
print(f"  Untagged/excluded: {int((raw['label'] == -1).sum())}")

untagged = raw[raw['label'] == -1].copy()
untagged['rule_tag'] = apply_rules(untagged)
untagged['label'] = untagged['rule_tag'].map(PLACE_TO_LABEL).fillna(-1).astype(int)
rule_tagged = untagged[untagged['label'] != -1].copy()

train_df = pd.concat([train_df, rule_tagged], ignore_index=True)

print(f"\nAfter rule augmentation:")
for lbl, place in sorted(LABEL_TO_PLACE.items()):
    n = int((train_df['label'] == lbl).sum())
    if n: print(f"  {place}: {n}")
print(f"  Total training rows: {len(train_df)}")


Multi-label rows excluded: 10
Priority Place
Eastern Pacific,SW Pacific + Indonesia, SWIO               4
Amazon, SWIO                                               3
SWIO, Eastern Pacific                                      2
Indonesia and Southern Pacific Islands, Greater Mekong     1

Pre-tagged label distribution:
  Amazon: 49
  Arctic: 15
  Congo Basin: 17
  Eastern Himalayas: 28
  Eastern Pacific Seascape: 8
  Great Plains: 84
  Greater Mekong: 15
  Southern Africa: 38
  Southwest Indian Ocean: 2
  SW Pacific + Indonesia: 16
  Untagged/excluded: 604

After rule augmentation:
  Amazon: 83
  Arctic: 16
  Congo Basin: 18
  Eastern Himalayas: 38
  Eastern Pacific Seascape: 9
  Great Plains: 92
  Greater Mekong: 17
  Southern Africa: 43
  Southwest Indian Ocean: 6
  SW Pacific + Indonesia: 18
  Total training rows: 340


### 4b · Augment with Actuals (multi-year)

In [5]:
# Add filenames here as new fiscal years become available.
# Files not found on disk are skipped automatically.
ACTUALS_FILES = [
    'Actuals_FY25(FY25 2026-03-09 12_11 AKDT).csv',  # FY25 — confirmed present
    # 'Actuals_FY24.csv',  # add when available
    # 'Actuals_FY23.csv',  # add when available
    # 'Actuals_FY22.csv',  # add when available
]

import os

GR_NAME_MAP = {}
for v in raw['Grant'].dropna().unique():
    code = str(v).split()[0]
    GR_NAME_MAP[code] = str(v).strip()

all_actuals_frames = []

for fname in ACTUALS_FILES:
    if not os.path.exists(fname):
        print(f"  [skip] {fname} — file not found")
        continue
    _hdr = pd.read_csv(fname, encoding='latin1', header=None, nrows=20)
    cc_map = {}
    for entry in str(_hdr.iloc[9, 1]).split('\n\n'):
        entry = entry.strip()
        parts = entry.split(' ', 1)
        if len(parts) == 2:
            cc_map[parts[0]] = entry
    df = pd.read_csv(fname, encoding='latin1', header=19, low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'Program': 'Program Code'})
    df['Cost Center'] = df['Cost Center'].astype(str).str.strip().map(lambda c: cc_map.get(c, c))
    df['Grant']       = df['Grant'].astype(str).str.strip().map(lambda g: GR_NAME_MAP.get(g, g))
    all_actuals_frames.append(df[['Cost Center', 'Program Code', 'Grant']])
    print(f"  [ok]   {fname} — {len(df):,} rows loaded")

if all_actuals_frames:
    combined      = pd.concat(all_actuals_frames, ignore_index=True)
    actuals_dedup = combined.drop_duplicates().copy()
    actuals_dedup['rule_tag'] = apply_rules(actuals_dedup)
    actuals_dedup['label']    = actuals_dedup['rule_tag'].map(PLACE_TO_LABEL).fillna(-1).astype(int)
    actuals_tagged = actuals_dedup[actuals_dedup['label'] != -1].copy()
    train_df = pd.concat([train_df, actuals_tagged], ignore_index=True)
    print(f"\nActuals summary:")
    print(f"  Files processed         : {len(all_actuals_frames)} of {len(ACTUALS_FILES)}")
    print(f"  Unique CC/Program/Grant : {len(actuals_dedup):,}")
    print(f"  Rule-tagged breakdown:")
    for lbl, place in sorted(LABEL_TO_PLACE.items()):
        n = int((actuals_tagged['label'] == lbl).sum())
        if n: print(f"    {place}: {n}")
    print(f"  Total training rows after Actuals: {len(train_df):,}")
else:
    print("No Actuals files found — training continues without Actuals augmentation.")


  [ok]   Actuals_FY25(FY25 2026-03-09 12_11 AKDT).csv — 503,386 rows loaded

Actuals summary:
  Files processed         : 1 of 1
  Unique CC/Program/Grant : 3,978
  Rule-tagged breakdown:
    Amazon: 292
    Arctic: 65
    Congo Basin: 44
    Eastern Himalayas: 84
    Eastern Pacific Seascape: 22
    Great Plains: 240
    Greater Mekong: 51
    Southern Africa: 55
    Southwest Indian Ocean: 7
    SW Pacific + Indonesia: 26
  Total training rows after Actuals: 1,226


### 5 · Pipeline 2 — ML Model

In [6]:
TEXT_COLS = SEGMENT_ORDER + ['Country', 'Big Bet']

def make_text(row):
    parts = [str(row.get(c, '') or '') for c in TEXT_COLS]
    return ' '.join(p for p in parts if p and p.lower() not in ('nan', ''))

train_df['text'] = train_df.apply(make_text, axis=1)
X = list(train_df['text'])
y = list(train_df['label'].astype(int))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ml_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2), min_df=1, max_features=50000, sublinear_tf=True,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=42,
    )),
])
ml_pipe.fit(X_train, y_train)
print('Model trained.')


Model trained.


### 6 · Evaluate ML Pipeline

In [7]:
y_pred = ml_pipe.predict(X_test)

present_labels = sorted(set(y_test) | set(y_pred))
present_names  = [LABEL_TO_PLACE[l] for l in present_labels]

print(f'Test accuracy : {accuracy_score(y_test, y_pred)*100:.1f}%')
print()
print(classification_report(y_test, y_pred, labels=present_labels,
      target_names=present_names, zero_division=0))

from sklearn.base import clone
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_arr, y_arr = np.array(X), np.array(y)
fold_accs = []
for tr_idx, te_idx in cv.split(X_arr, y_arr):
    p = clone(ml_pipe)
    p.fit(X_arr[tr_idx], y_arr[tr_idx])
    fold_accs.append(accuracy_score(y_arr[te_idx], p.predict(X_arr[te_idx])))

print(f'CV scores : {[f"{a*100:.1f}%" for a in fold_accs]}')
print(f'CV mean   : {np.mean(fold_accs)*100:.1f}%  +/-  {np.std(fold_accs)*100:.1f}%')
print(f'Target    : 80.0%  ->  {"PASS!" if np.mean(fold_accs) >= 0.80 else "FAIL :("}')


Test accuracy : 99.2%

                          precision    recall  f1-score   support

                  Amazon       0.99      0.99      0.99        75
                  Arctic       1.00      1.00      1.00        16
             Congo Basin       1.00      1.00      1.00        12
       Eastern Himalayas       1.00      0.96      0.98        24
Eastern Pacific Seascape       1.00      1.00      1.00         6
            Great Plains       1.00      1.00      1.00        67
          Greater Mekong       1.00      1.00      1.00        14
         Southern Africa       1.00      1.00      1.00        20
  Southwest Indian Ocean       1.00      1.00      1.00         3
  SW Pacific + Indonesia       0.90      1.00      0.95         9

                accuracy                           0.99       246
               macro avg       0.99      0.99      0.99       246
            weighted avg       0.99      0.99      0.99       246

CV scores : ['99.6%', '99.6%', '97.6%', '98.8%', '

### 7 · Apply Both Pipelines to All Rows

In [8]:
raw['text']     = raw.apply(make_text, axis=1)
raw['rule_tag']  = apply_rules(raw)
raw['ml_tag']    = ml_pipe.predict(raw['text'].tolist())
raw['ml_tag']    = raw['ml_tag'].map(LABEL_TO_PLACE)

# Rule-first: trust keyword match; fall back to ML only when rules find nothing
raw['final_tag'] = raw.apply(
    lambda r: r['rule_tag'] if pd.notna(r['rule_tag']) else r['ml_tag'], axis=1
)

raw['rule_match'] = raw['rule_tag'].notna()

print("Tagged row counts:")
for place in CANONICAL_PLACES:
    print(f"  {place}: {int((raw['final_tag'] == place).sum())}")

print(f"\nRule match (keyword found) : {raw['rule_match'].sum()}")
print(f"ML only (no keyword match) : {(~raw['rule_match']).sum()}")

labeled = raw[raw['label'] != -1].copy()
y_true      = labeled['label'].astype(int).tolist()
y_pred_comb = labeled['final_tag'].map(PLACE_TO_LABEL).fillna(-1).astype(int).tolist()

present_labels = sorted(set(y_true) | set(y_pred_comb))
present_names  = [LABEL_TO_PLACE.get(l, 'Unknown') for l in present_labels]
print(f'\nAccuracy on originally labeled rows : {accuracy_score(y_true, y_pred_comb)*100:.1f}%')
print(classification_report(y_true, y_pred_comb, labels=present_labels,
      target_names=present_names, zero_division=0))


Tagged row counts:
  Amazon: 583
  Arctic: 23
  Congo Basin: 20
  Eastern Himalayas: 38
  Eastern Pacific Seascape: 10
  Great Plains: 107
  Greater Mekong: 18
  Southern Africa: 47
  Southwest Indian Ocean: 6
  SW Pacific + Indonesia: 24

Rule match (keyword found) : 269
ML only (no keyword match) : 607

Accuracy on originally labeled rows : 100.0%
                          precision    recall  f1-score   support

                  Amazon       1.00      1.00      1.00        49
                  Arctic       1.00      1.00      1.00        15
             Congo Basin       1.00      1.00      1.00        17
       Eastern Himalayas       1.00      1.00      1.00        28
Eastern Pacific Seascape       1.00      1.00      1.00         8
            Great Plains       1.00      1.00      1.00        84
          Greater Mekong       1.00      1.00      1.00        15
         Southern Africa       1.00      1.00      1.00        38
  Southwest Indian Ocean       1.00      1.00      1.

In [9]:
validation_df = raw.copy()

probas = ml_pipe.predict_proba(validation_df['text'].tolist())
validation_df['confidence'] = probas.max(axis=1).round(3)

export_cols = ['Cost Center', 'Program Code', 'Grant', 'Country', 'Big Bet',
               'Priority Place', 'final_tag', 'confidence',
               'rule_match', 'multi_label']
export_cols = [c for c in export_cols if c in validation_df.columns]

snapshot = validation_df[export_cols].rename(columns={
    'Priority Place':  'original_label',
    'final_tag':       'predicted_label',
    'rule_match':      'ml_classified',
    'multi_label':     'excluded_multi_label',
})

# ml_classified: True = no keyword match, ML was sole classifier
#                False = rule pipeline found a keyword match
snapshot['ml_classified'] = ~snapshot['ml_classified']

snapshot['disagrees'] = pd.array([pd.NA] * len(snapshot), dtype=object)
has_original = snapshot['original_label'].notna() & ~snapshot['excluded_multi_label']
original_canonical = raw.loc[has_original, 'Priority Place'].str.strip().map(PLACE_MAPPING)
snapshot.loc[has_original, 'disagrees'] = (
    original_canonical != snapshot.loc[has_original, 'predicted_label']
).values

snapshot.to_csv('validation_snapshot.csv', index=False)

print(f"Exported {len(snapshot):,} rows to validation_snapshot.csv")
print(f"  ml_classified=True  (ML only)      : {snapshot['ml_classified'].sum()}")
print(f"  ml_classified=False (rule match)   : {(~snapshot['ml_classified']).sum()}")
print(f"  Disagreements (model vs hand label): {int(snapshot['disagrees'].sum())}")
print(f"  Multi-label (excluded)             : {int(snapshot['excluded_multi_label'].sum())}")
snapshot[snapshot['disagrees'] == True].head(10)


Exported 876 rows to validation_snapshot.csv
  ml_classified=True  (ML only)      : 607
  ml_classified=False (rule match)   : 269
  Disagreements (model vs hand label): 0
  Multi-label (excluded)             : 10


,Cost Center,Program Code,Grant,Country,Big Bet,original_label,predicted_label,confidence,ml_classified,excluded_multi_label,disagrees


In [10]:
raw['text']     = raw.apply(make_text, axis=1)
raw['rule_tag']  = raw.apply(
    lambda r: rule_scan_row(r, segment_order=INFERENCE_SEGMENT_ORDER), axis=1
)
raw['ml_tag']    = ml_pipe.predict(raw['text'].tolist())
raw['ml_tag']    = raw['ml_tag'].map(LABEL_TO_PLACE)

# Rule-first: trust keyword match; fall back to ML only when rules find nothing
raw['final_tag'] = raw.apply(
    lambda r: r['rule_tag'] if pd.notna(r['rule_tag']) else r['ml_tag'], axis=1
)

raw['rule_match'] = raw['rule_tag'].notna()

print("Tagged row counts:")
for place in CANONICAL_PLACES:
    print(f"  {place}: {int((raw['final_tag'] == place).sum())}")

print(f"\nRule match (keyword found) : {raw['rule_match'].sum()}")
print(f"ML only (no keyword match) : {(~raw['rule_match']).sum()}")

labeled = raw[raw['label'] != -1].copy()
y_true      = labeled['label'].astype(int).tolist()
y_pred_comb = labeled['final_tag'].map(PLACE_TO_LABEL).fillna(-1).astype(int).tolist()

present_labels = sorted(set(y_true) | set(y_pred_comb))
present_names  = [LABEL_TO_PLACE.get(l, 'Unknown') for l in present_labels]
print(f'\nAccuracy on originally labeled rows : {accuracy_score(y_true, y_pred_comb)*100:.1f}%')
print(classification_report(y_true, y_pred_comb, labels=present_labels,
      target_names=present_names, zero_division=0))


Tagged row counts:
  Amazon: 580
  Arctic: 23
  Congo Basin: 20
  Eastern Himalayas: 38
  Eastern Pacific Seascape: 10
  Great Plains: 107
  Greater Mekong: 19
  Southern Africa: 47
  Southwest Indian Ocean: 8
  SW Pacific + Indonesia: 24

Rule match (keyword found) : 284
ML only (no keyword match) : 592

Accuracy on originally labeled rows : 100.0%
                          precision    recall  f1-score   support

                  Amazon       1.00      1.00      1.00        49
                  Arctic       1.00      1.00      1.00        15
             Congo Basin       1.00      1.00      1.00        17
       Eastern Himalayas       1.00      1.00      1.00        28
Eastern Pacific Seascape       1.00      1.00      1.00         8
            Great Plains       1.00      1.00      1.00        84
          Greater Mekong       1.00      1.00      1.00        15
         Southern Africa       1.00      1.00      1.00        38
  Southwest Indian Ocean       1.00      1.00      1.

### 8 · Top TF-IDF Features by Importance

In [ ]:
feature_names = ml_pipe.named_steps['tfidf'].get_feature_names_out()
importances   = ml_pipe.named_steps['rf'].feature_importances_

feat_df = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(feat_df['feature'][::-1], feat_df['importance'][::-1], color=WWF_GREEN)
ax.set_xlabel('Random Forest Feature Importance')
ax.set_title('Top 20 TF-IDF Features — 10-Class Classifier', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print(feat_df.to_string(index=False))
